#Lectura de datos ECOBICI
En este script pasan varias cosas:
1. Se consulta y descarga la información de datos abiertos de ecobici.
2. Se generan archivos de .parquet con esos datos.
3. Se consolida un .csv con resúmenes de viajes por estación y por edades las transacciones ocurridas de 2020 a 2026.
4. Se agrupan por alcaldía.


In [24]:
import re
import io
import unicodedata
import time
from urllib.parse import urljoin
from dataclasses import dataclass, field
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Con esto leemos los datos de las URLs de los CSV históricos
Leemos de lapágina oficial y extraemos TODOS los enlaces que terminan en .csv
Con este URL "https://ecobici.cdmx.gob.mx/datos-abiertos/"





In [26]:
URL_PAGINA = "https://ecobici.cdmx.gob.mx/datos-abiertos/"

ANIO_MIN = 2020
ANIO_MAX = 2026

def obtener_urls_csv(url_pagina: str) -> pd.DataFrame:
    """
    Descarga el HTML de la página de datos abiertos de Ecobici y extrae
    todos los enlaces .csv encontrados, junto con el año/mes que se puede
    inferir del nombre del archivo o del texto del enlace.

    Devuelve un DataFrame con columnas: ['anio', 'mes', 'url', 'nombre_archivo']
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; EcobiciDataReader/1.0)"}
    resp = requests.get(url_pagina, headers=headers, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")

    registros = []
    for a in soup.find_all("a", href=True):
        href_original = a["href"]
        if href_original.lower().endswith(".csv"):
            # Algunos enlaces vienen como ruta relativa (ej. "/wp-content/...")
            # en vez de URL completa. urljoin() los convierte siempre en una
            # URL absoluta usando la página como referencia; si el href ya
            # era absoluto, lo deja igual.
            href = urljoin(url_pagina, href_original)

            texto_enlace = a.get_text(strip=True)  # normalmente algo como "2024-02"

            # Intentamos inferir año y mes primero del TEXTO del enlace
            # (más confiable, ej. "2024-02") y si falla, del nombre de archivo.
            match = re.search(r"(20\d{2})[-_]?(\d{2})", texto_enlace) or \
                    re.search(r"(20\d{2})[-_]?(\d{2})", href)

            anio = int(match.group(1)) if match else None
            mes = int(match.group(2)) if match else None

            registros.append({
                "anio": anio,
                "mes": mes,
                "url": href,
                "nombre_archivo": href.split("/")[-1],
            })

    df_urls = pd.DataFrame(registros).drop_duplicates(subset="url").reset_index(drop=True)
    return df_urls


df_urls = obtener_urls_csv(URL_PAGINA)

# Filtramos por el rango de años deseado y quitamos filas sin año detectado
df_urls = df_urls[
    df_urls["anio"].between(ANIO_MIN, ANIO_MAX, inclusive="both")
].sort_values(["anio", "mes"]).reset_index(drop=True)

print(f"Se encontraron {len(df_urls)} archivos CSV entre {ANIO_MIN} y {ANIO_MAX}.")
df_urls.head(15)


Se encontraron 81 archivos CSV entre 2020 y 2026.


,anio,mes,url,nombre_archivo
0,2020,1,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-01.csv
1,2020,2,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-02.csv
2,2020,3,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-03.csv
3,2020,4,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-04.csv
4,2020,5,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-05.csv
5,2020,6,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-06.csv
6,2020,7,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-07.csv
7,2020,8,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-08.csv
8,2020,9,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-09.csv
9,2020,10,https://ecobici.cdmx.gob.mx/wp-content/uploads...,2020-10.csv


In [27]:
# ETAPA 2: Función para leer un CSV remoto DIRECTAMENTE EN MEMORIA
# -----------------------------------------------------------------------
# Nunca se escribe el archivo a disco: se descargan los bytes con requests
# y se pasan a pandas a través de un buffer en memoria (io.BytesIO).

# Los archivos históricos de Ecobici no son 100% homogéneos:
#   - Algunos vienen en UTF-8, otros en Latin-1 / CP1252.
#   - Algunos usan "," como separador, otros ";".
# Por eso la función intenta varias combinaciones antes de rendirse.

def leer_csv_remoto(
    url: str,
    timeout: int = 60,
    reintentos: int = 5,
    espera_base_seg: int = 10,
) -> pd.DataFrame | None:
    """
    Descarga un CSV desde `url` y lo carga en memoria como DataFrame,
    sin guardarlo en disco. Prueba distintas codificaciones/separadores
    comunes en los archivos históricos de Ecobici.

    Incluye reintentos con espera creciente (backoff exponencial) para
    tolerar fallas de red pasajeras -- importante en una corrida larga y
    sin supervisión (ej. dejarla corriendo toda la noche).

    Devuelve el DataFrame, o None si no fue posible leerlo tras agotar
    los reintentos (y lo reporta).
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; EcobiciDataReader/1.0)"}

    contenido = None
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            resp.raise_for_status()
            contenido = resp.content  # bytes crudos, viven solo en memoria (RAM)
            break
        except requests.RequestException as e:
            if intento == reintentos:
                print(f"  [ERROR] Agotados los {reintentos} intentos para {url}: {e}")
                return None
            espera = espera_base_seg * intento  # 10s, 20s, 30s, 40s...
            print(f"  [AVISO] Falló intento {intento}/{reintentos} para {url} "
                  f"({e}). Reintentando en {espera}s...")
            time.sleep(espera)

    intentos_parseo = [
        {"encoding": "utf-8", "sep": ","},
        {"encoding": "utf-8", "sep": ";"},
        {"encoding": "latin-1", "sep": ","},
        {"encoding": "latin-1", "sep": ";"},
    ]

    for opciones in intentos_parseo:
        try:
            buffer = io.BytesIO(contenido)  # buffer en memoria, no en disco
            df = pd.read_csv(
                buffer,
                encoding=opciones["encoding"],
                sep=opciones["sep"],
                low_memory=False,
                on_bad_lines="skip",  # ignora filas corruptas en vez de romper
            )
            # Heurística mínima: si solo detectó 1 columna, el separador
            # probablemente estaba mal -> probamos la siguiente combinación.
            if df.shape[1] > 1:
                return df
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue

    print(f"  [ERROR] No se pudo parsear correctamente: {url}")
    return None


In [28]:
# Configurar logging a archivo (para revisar el avance sin abrir Colab)
# -----------------------------------------------------------------------
# Todo lo que antes era "print" ahora también queda escrito en un archivo
# de texto en tu Drive. Así, en la mañana, puedes abrir ese archivo (o
# incluso revisarlo desde el celular con la app de Drive) sin necesidad de
# volver a abrir el notebook.

import os
import json
import logging

LOG_PATH = "/content/drive/MyDrive/ecobici_log.txt"

logger = logging.getLogger("ecobici")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # evita duplicar líneas si vuelves a correr esta celda

formato = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

manejador_archivo = logging.FileHandler(LOG_PATH)
manejador_archivo.setFormatter(formato)
logger.addHandler(manejador_archivo)

manejador_consola = logging.StreamHandler()
manejador_consola.setFormatter(formato)
logger.addHandler(manejador_consola)

logger.info("=== Nueva corrida iniciada ===")
print(f"El log se está guardando en: {LOG_PATH}")

# ETAPA 3: Normalización de columnas (definida ANTES de descargar)
# -----------------------------------------------------------------------
# IMPORTANTE: después de correr la Celda 6 (inspección de columnas), revisa
# si aparecen variantes nuevas y agrégalas aquí a MAPA_SINONIMOS.

def normalizar_texto(texto: str) -> str:
    """Minúsculas, sin acentos, espacios -> guion_bajo."""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", "_", texto).strip("_")
    return texto


# Diccionario de sinónimos -> nombre estándar final.
MAPA_SINONIMOS = {
    "genero_usuario": "genero",
    "genero_ret": "genero",
    "bici": "bici_retiro",
    "id_bici": "bici_retiro",
    "ciclo_estacion_retiro": "ciclo_estacion_retiro",
    "ciclosestacion_retiro": "ciclo_estacion_retiro",
    "ciclo_estacion_arribo": "ciclo_estacion_arribo",
    "ciclosestacion_arribo": "ciclo_estacion_arribo",
    "fecha_retiro": "fecha_retiro",
    "fecha_arribo": "fecha_arribo",
    "hora_retiro": "hora_retiro",
    "hora_arribo": "hora_arribo",
    "edad_usuario": "edad",
}


def estandarizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica normalizar_texto() + MAPA_SINONIMOS a todas las columnas."""
    nuevas_columnas = {}
    for col in df.columns:
        col_norm = normalizar_texto(col)
        col_final = MAPA_SINONIMOS.get(col_norm, col_norm)
        nuevas_columnas[col] = col_final
    return df.rename(columns=nuevas_columnas)


# ETAPA 4 - FASE A: Descargar y normalizar cada mes, con CHECKPOINT en disco
# -----------------------------------------------------------------------
# Diseño pensado para correr sin supervisión (ej. toda la noche):
#
# 1) Cada mes, ya normalizado, se guarda como un archivo .parquet individual
#    en CARPETA_TEMP (uno por mes). Parquet conserva tipos de dato y pesa
#    menos que CSV.
# 2) CHECKPOINT: si el .parquet de un mes ya existe, ese mes se OMITE.
# 3) Cada descarga tiene reintentos con backoff (ver leer_csv_remoto), y
#    cualquier error inesperado en un mes se registra en el log y NO
#    detiene la corrida completa -- sigue con el siguiente mes.
# 4) Solo se mantiene UN mes en memoria a la vez (se libera con `del`).

CARPETA_TEMP = "/content/drive/MyDrive/ecobici_tmp_meses"
os.makedirs(CARPETA_TEMP, exist_ok=True)

archivos_fallidos = []

for i, fila in df_urls.iterrows():
    anio, mes, url = fila["anio"], fila["mes"], fila["url"]
    clave = f"{anio}-{mes:02d}"
    ruta_tmp = os.path.join(CARPETA_TEMP, f"{clave}.parquet")

    # --- CHECKPOINT: saltar meses ya procesados en una corrida anterior ---
    if os.path.exists(ruta_tmp):
        logger.info(f"[{i+1}/{len(df_urls)}] {clave}: ya existe (checkpoint), se omite.")
        continue

    logger.info(f"[{i+1}/{len(df_urls)}] Procesando {clave} -> {url}")

    try:
        df_mes = leer_csv_remoto(url)

        if df_mes is None or df_mes.empty:
            archivos_fallidos.append({"anio": anio, "mes": mes, "url": url})
            logger.warning(f"{clave}: no se pudo leer / venía vacío, se omite.")
            continue

        # Etiquetamos el origen y normalizamos columnas.
        df_mes["anio_archivo"] = anio
        df_mes["mes_archivo"] = mes
        df_mes["archivo_origen"] = fila["nombre_archivo"]
        df_mes = estandarizar_columnas(df_mes)

        # Guardamos el checkpoint del mes. Si la sesión se corta justo
        # después de esta línea, el mes ya quedó a salvo en Drive.
        df_mes.to_parquet(ruta_tmp, index=False)

        logger.info(f"{clave}: OK ({df_mes.shape[0]:,} filas, {df_mes.shape[1]} columnas).")
        del df_mes

    except Exception as e:
        # Un error inesperado en UN mes no debe tumbar toda la corrida
        # nocturna -- lo registramos y seguimos.
        logger.error(f"{clave}: error inesperado -> {e}")
        archivos_fallidos.append({"anio": anio, "mes": mes, "url": url, "error": str(e)})

    time.sleep(0.3)  # pequeña pausa para no saturar el servidor de Ecobici

logger.info(f"FASE A terminada. Checkpoints listos: {len(os.listdir(CARPETA_TEMP))} "
            f"de {len(df_urls)} meses. Fallidos en esta corrida: {len(archivos_fallidos)}")

if archivos_fallidos:
    logger.warning("Meses fallidos en esta corrida (vuelve a correr esta celda para reintentarlos):")
    for a in archivos_fallidos:
        logger.warning(f"  {a['anio']}-{a['mes']:02d}: {a['url']}")

2026-10-06 21:12:04,347 | INFO | === Nueva corrida iniciada ===
INFO:ecobici:=== Nueva corrida iniciada ===
2026-10-06 21:12:04,373 | INFO | [1/81] 2020-01: ya existe (checkpoint), se omite.
INFO:ecobici:[1/81] 2020-01: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,378 | INFO | [2/81] 2020-02: ya existe (checkpoint), se omite.
INFO:ecobici:[2/81] 2020-02: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,381 | INFO | [3/81] 2020-03: ya existe (checkpoint), se omite.
INFO:ecobici:[3/81] 2020-03: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,385 | INFO | [4/81] 2020-04: ya existe (checkpoint), se omite.
INFO:ecobici:[4/81] 2020-04: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,387 | INFO | [5/81] 2020-05: ya existe (checkpoint), se omite.
INFO:ecobici:[5/81] 2020-05: ya existe (checkpoint), se omite.


El log se está guardando en: /content/drive/MyDrive/ecobici_log.txt


2026-10-06 21:12:04,390 | INFO | [6/81] 2020-06: ya existe (checkpoint), se omite.
INFO:ecobici:[6/81] 2020-06: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,393 | INFO | [7/81] 2020-07: ya existe (checkpoint), se omite.
INFO:ecobici:[7/81] 2020-07: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,409 | INFO | [8/81] 2020-08: ya existe (checkpoint), se omite.
INFO:ecobici:[8/81] 2020-08: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,424 | INFO | [9/81] 2020-09: ya existe (checkpoint), se omite.
INFO:ecobici:[9/81] 2020-09: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,435 | INFO | [10/81] 2020-10: ya existe (checkpoint), se omite.
INFO:ecobici:[10/81] 2020-10: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,440 | INFO | [11/81] 2020-11: ya existe (checkpoint), se omite.
INFO:ecobici:[11/81] 2020-11: ya existe (checkpoint), se omite.
2026-10-06 21:12:04,446 | INFO | [12/81] 2020-12: ya existe (checkpoint), se omite.
INFO:ecobici:[12/81] 2020-12: ya exi

#Revisión de estructura de un un .parquet
Para este punto tenemos los archivos necesarios en Drive, en .parquet (Las celdas antetiores fueron ejecutados en diferentes bloques para lograr la conversión completa, solo ajustando los rangos de años).

In [29]:
# =============================================================================
# ECOBICI - ANÁLISIS DE UN SOLO ARCHIVO .PARQUET
# =============================================================================
# Carga y analiza a fondo UN archivo .parquet de la carpeta de checkpoints:
# estructura, tipos de dato, nulos, valores únicos y muestra de filas.
# Cargar un solo mes es liviano, así que no hay riesgo de memoria.
# =============================================================================

# ETAPA 0: Elegir y cargar el archivo
# -----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os
import pandas as pd

CARPETA_TEMP = "/content/drive/MyDrive/ecobici_tmp_meses"

# Cambia este nombre por el archivo que quieras analizar (ej. "2022-06.parquet").
# Si lo dejas como None, se toma automáticamente el primero que encuentre.
NOMBRE_ARCHIVO = "2026-02.parquet"

archivos_disponibles = sorted(f for f in os.listdir(CARPETA_TEMP) if f.endswith(".parquet"))
if not archivos_disponibles:
    raise FileNotFoundError(f"No hay archivos .parquet en {CARPETA_TEMP}")

archivo_elegido = NOMBRE_ARCHIVO or archivos_disponibles[0]
ruta = os.path.join(CARPETA_TEMP, archivo_elegido)

df = pd.read_parquet(ruta)

print(f"Archivo analizado: {archivo_elegido}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo analizado: 2026-02.parquet
Filas: 1,528,610 | Columnas: 12


In [30]:
# ETAPA 1: Estructura -- nombres de columna y tipos de dato
# -----------------------------------------------------------------------
print("Columnas y tipos de dato:\n")
print(df.dtypes)
# ETAPA 2: Muestra de filas
# ----------------------------------------------------------------------
df.head(10)

# ETAPA 3: Valores nulos por columna
# -----------------------------------------------------------------------

print("Nulos por columna (cantidad y %):\n")
resumen_nulos = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
})
print(resumen_nulos.sort_values("pct_nulos", ascending=False))

Columnas y tipos de dato:

genero                    object
edad                     float64
bici_retiro                int64
ciclo_estacion_retiro     object
fecha_retiro              object
hora_retiro               object
ciclo_estacionarribo      object
fecha_arribo              object
hora_arribo               object
anio_archivo               int64
mes_archivo                int64
archivo_origen            object
dtype: object
Nulos por columna (cantidad y %):

                       n_nulos  pct_nulos
genero                       0        0.0
edad                        68        0.0
bici_retiro                  0        0.0
ciclo_estacion_retiro        0        0.0
fecha_retiro                 0        0.0
hora_retiro                  0        0.0
ciclo_estacionarribo         0        0.0
fecha_arribo                 0        0.0
hora_arribo                  0        0.0
anio_archivo                 0        0.0
mes_archivo                  0        0.0
archivo_origen          

In [31]:
df.head(10)

,genero,edad,bici_retiro,ciclo_estacion_retiro,fecha_retiro,hora_retiro,ciclo_estacionarribo,fecha_arribo,hora_arribo,anio_archivo,mes_archivo,archivo_origen
0,?,31.0,8815912,259,31/01/2026,23:52:52,545,01/02/2026,00:00:10,2026,2,2026-02.csv
1,M,19.0,5796800,145,31/01/2026,23:46:59,171,01/02/2026,00:00:16,2026,2,2026-02.csv
2,M,47.0,2022705,068,31/01/2026,23:57:09,078,01/02/2026,00:00:20,2026,2,2026-02.csv
3,M,31.0,3139730,127,31/01/2026,23:46:19,546,01/02/2026,00:00:23,2026,2,2026-02.csv
4,M,22.0,5941455,302,31/01/2026,23:43:38,136,01/02/2026,00:00:33,2026,2,2026-02.csv
5,F,29.0,4960266,546,31/01/2026,23:55:06,538,01/02/2026,00:00:37,2026,2,2026-02.csv
6,?,21.0,2054377,508,31/01/2026,23:57:30,512,01/02/2026,00:00:41,2026,2,2026-02.csv
7,M,25.0,5314471,069,31/01/2026,23:47:55,137,01/02/2026,00:01:02,2026,2,2026-02.csv
8,M,32.0,3328242,614,31/01/2026,23:50:11,606,01/02/2026,00:01:05,2026,2,2026-02.csv
9,M,54.0,2339968,179,31/01/2026,23:53:12,165,01/02/2026,00:01:06,2026,2,2026-02.csv


In [32]:
# ETAPA 5: Valores únicos por columna (útil para detectar columnas categóricas
# -----------------------------------------------------------------------

MAX_VALORES_A_MOSTRAR = 15  # si una columna tiene menos categorías que esto, se listan

for col in df.columns:
    n_unicos = df[col].nunique(dropna=True)
    print(f"- {col}: {n_unicos} valores únicos")
    if n_unicos <= MAX_VALORES_A_MOSTRAR:
        print(f"    {sorted(df[col].dropna().unique().tolist())}")

- genero: 4 valores únicos
    ['?', 'F', 'M', 'O']
- edad: 74 valores únicos
- bici_retiro: 7228 valores únicos
- ciclo_estacion_retiro: 677 valores únicos
- fecha_retiro: 37 valores únicos
- hora_retiro: 69688 valores únicos
- ciclo_estacionarribo: 677 valores únicos
- fecha_arribo: 28 valores únicos
- hora_arribo: 70792 valores únicos
- anio_archivo: 1 valores únicos
    [2026]
- mes_archivo: 1 valores únicos
    [2]
- archivo_origen: 1 valores únicos
    ['2026-02.csv']


# Viajes por estación
Objetivo: para cada archivo .parquet dentro del rango de años elegido,
contar cuántos viajes se iniciaron en cada estación y calcular el
promedio de edad, separados por género (H/M), y escribir lo en un CSV -- SIN cargar todos los parquets juntos en RAM.

In [33]:
# =============================================================================
# ECOBICI - RESUMEN POR TRIMESTRE Y ALCALDÍA + ESTACIONES ACTIVAS




#   - short_name  -> el número de estación "clásico" (ej. 508, 509, 510...),
#                    que SÍ corresponde a ciclo_estacion_retiro en los viajes.
# =============================================================================
# CORRECCIÓN respecto a la versión anterior: el cruce estación -> alcaldía
# fallaba (clasificaba mal / dejaba como SIN_CATALOGO de más) porque:
#
#   1) El catálogo convertía "short_name" a int limpio, pero
#      "ciclo_estacion_retiro" en los parquet puede venir como texto con
#      ceros a la izquierda ("027"), como float ("27.0") o con otro
#      formato -- un dict con llaves int no encuentra esas variantes.
#   2) Si dos estaciones distintas comparten el mismo "short_name" en el
#      catálogo, dict(zip(...)) se queda silenciosamente con la última y
#      pisa a la otra.
#
# Ahora ambos lados del cruce pasan por la MISMA función de normalización
# (normalizar_num_estacion), que extrae solo los dígitos y quita ceros a
# la izquierda, así que "027", 27, 27.0 y "27" terminan siendo la misma
# llave. Además se reporta si hay duplicados en el catálogo y, al final,
# un diagnóstico de cuántos viajes quedaron sin alcaldía conocida.
#
# Todo lo demás (diseño de memoria por trimestre, encabezados de columnas,
# nombres de archivo) se mantiene igual que la versión anterior.
# =============================================================================

# ETAPA 0: Imports y configuración
# -----------------------------------------------------------------------

import os
import re
import glob
import json
import unicodedata
import pandas as pd
import pyarrow.parquet as pq

CARPETA_TEMP = "/content/drive/MyDrive/ecobici_tmp_meses"
RUTA_SALIDA = "/content/drive/MyDrive/Tesis_caso/ecobici_resumen_viajes_trimestre_alcaldia.csv"
ESTADO_RESUMEN = "/content/drive/MyDrive/Tesis_caso/ecobici_estado_resumen_trimestre_alcaldia.json"

# --- Intervalo de años a procesar: AJUSTA AQUÍ cuando quieras otro rango ---
ANIO_MIN = 2020
ANIO_MAX = 2026

# Nombres LÓGICOS que usaremos, independientemente de cómo se llame la
# columna real en cada archivo (ya NO incluye edad).
COL_ESTACION_ORIGEN = "ciclo_estacion_retiro"
COL_GENERO = "genero"
COLUMNAS_NECESARIAS = [COL_ESTACION_ORIGEN, COL_GENERO]

# Mismo diccionario de sinónimos usado siempre en este proyecto.
MAPA_SINONIMOS = {
    "genero_usuario": "genero",
    "genero_ret": "genero",
    "bici": "bici_retiro",
    "id_bici": "bici_retiro",
    "ciclo_estacion_retiro": "ciclo_estacion_retiro",
    "ciclosestacion_retiro": "ciclo_estacion_retiro",
    "estacion_retiro": "ciclo_estacion_retiro",
    "ce_retiro": "ciclo_estacion_retiro",
    "ciclo_estacion_arribo": "ciclo_estacion_arribo",
    "ciclosestacion_arribo": "ciclo_estacion_arribo",
    "estacion_arribo": "ciclo_estacion_arribo",
    "ce_arribo": "ciclo_estacion_arribo",
    "fecha_retiro": "fecha_retiro",
    "fecha_arribo": "fecha_arribo",
    "hora_retiro": "hora_retiro",
    "hora_arribo": "hora_arribo",
    "edad_usuario": "edad",
}


def normalizar_texto(texto: str) -> str:
    """Minúsculas, sin acentos, espacios -> guion_bajo."""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", "_", texto).strip("_")
    return texto


def normalizar_num_estacion(valor):
    """
    Lleva un número de estación a una forma canónica para poder cruzarlo
    sin importar cómo venga formateado en cada archivo: 27, 27.0, "27",
    "027" o "ECOBICI-027" deben mapear todos a la misma llave "27".
    Devuelve None si no se pudo extraer ningún dígito (valor vacío/NaN).
    """
    if pd.isna(valor):
        return None
    if isinstance(valor, float):
        valor = int(valor)          # 99.0 -> 99, ANTES de convertir a texto
    texto = str(valor).split(".")[0]  # por si llega como texto "99.0"
    digitos = re.sub(r"\D", "", texto)  # se queda solo con los dígitos
    if not digitos:
        return None
    return str(int(digitos))  # quita ceros a la izquierda, ej. "027" -> "27"


def resolver_columnas(ruta: str, columnas_logicas: list) -> tuple[dict, list]:
    """Lee solo el ESQUEMA de un parquet y mapea cada nombre lógico pedido
    a la columna real correspondiente en ese archivo (ver explicación en
    versiones anteriores de este script)."""
    columnas_reales = pq.ParquetFile(ruta).schema.names
    columna_real_por_logico = {}
    for col_real in columnas_reales:
        col_norm = normalizar_texto(col_real)
        col_logico = MAPA_SINONIMOS.get(col_norm, col_norm)
        columna_real_por_logico.setdefault(col_logico, col_real)

    resultado, faltantes = {}, []
    for logica in columnas_logicas:
        if logica in columna_real_por_logico:
            resultado[logica] = columna_real_por_logico[logica]
        else:
            faltantes.append(logica)
    return resultado, faltantes


def cargar_estado(ruta: str) -> set:
    if os.path.exists(ruta):
        with open(ruta, "r") as f:
            return set(json.load(f))
    return set()


def guardar_estado(ruta: str, estado: set) -> None:
    with open(ruta, "w") as f:
        json.dump(sorted(estado), f)


def anio_mes_desde_nombre(nombre_archivo: str) -> tuple[int, int]:
    """Extrae (año, mes) directo del nombre del checkpoint, ej. '2022-09.parquet'."""
    base = nombre_archivo.replace(".parquet", "")
    anio_str, mes_str = base.split("-")
    return int(anio_str), int(mes_str)


def mes_a_trimestre(mes: int) -> int:
    """Trimestre calendario estándar: ene-mar=T1, abr-jun=T2, jul-sep=T3, oct-dic=T4."""
    return (mes - 1) // 3 + 1


# ETAPA 1: Cargar el catálogo de estaciones -> alcaldía
# -----------------------------------------------------------------------
# Necesario para poder contar "estaciones activas por alcaldía" -- sin
# esto no sabríamos a qué alcaldía pertenece cada número de estación.
# Recuerda: el cruce correcto es por "short_name" (el número clásico de
# estación), NO por "station_id" (ID interno del feed).

coincidencias_catalogo = glob.glob("/content/drive/MyDrive/**/ecobici_estaciones_alcaldia.csv", recursive=True)
if not coincidencias_catalogo:
    raise FileNotFoundError("No encontré 'ecobici_estaciones_alcaldia.csv' en tu Drive.")

df_catalogo = pd.read_csv(coincidencias_catalogo[0])

# --- Normalización consistente del número de estación (la corrección clave) ---
df_catalogo["clave_estacion"] = df_catalogo["short_name"].map(normalizar_num_estacion)
filas_sin_clave = df_catalogo["clave_estacion"].isna().sum()
if filas_sin_clave:
    print(f"[AVISO] {filas_sin_clave} fila(s) del catálogo no tienen un 'short_name' "
          f"del que se pueda extraer un número de estación; se excluyen del cruce.")
df_catalogo = df_catalogo.dropna(subset=["clave_estacion"])

# --- Detectar y avisar de duplicados antes de que se pisen en silencio ---
duplicados = df_catalogo[df_catalogo.duplicated("clave_estacion", keep=False)]
if not duplicados.empty:
    n_claves_dup = duplicados["clave_estacion"].nunique()
    print(f"[AVISO] {n_claves_dup} número(s) de estación están duplicados en el catálogo "
          f"(distintas filas con el mismo short_name normalizado). Se conserva la primera "
          f"coincidencia de cada uno. Ejemplos:")
    print(duplicados.sort_values("clave_estacion")[["clave_estacion", "name", "alcaldia"]].head(10).to_string(index=False))

df_catalogo_unico = df_catalogo.drop_duplicates("clave_estacion", keep="first")

# Diccionario {clave_estacion_normalizada: alcaldia} -- lookup rápido, sin merges.
MAPA_ESTACION_ALCALDIA = dict(zip(df_catalogo_unico["clave_estacion"], df_catalogo_unico["alcaldia"]))

print(f"Catálogo cargado: {len(MAPA_ESTACION_ALCALDIA):,} estaciones con alcaldía conocida "
      f"(de {len(df_catalogo):,} filas originales).")


# ETAPA 2: Agrupar los archivos .parquet por (año, trimestre)
# -----------------------------------------------------------------------

archivos_disponibles = sorted(
    f for f in os.listdir(CARPETA_TEMP)
    if f.endswith(".parquet") and ANIO_MIN <= int(f.split("-")[0]) <= ANIO_MAX
)

grupos_por_trimestre = {}  # {(anio, trimestre): [nombres_de_archivo]}
for nombre in archivos_disponibles:
    anio, mes = anio_mes_desde_nombre(nombre)
    trimestre = mes_a_trimestre(mes)
    grupos_por_trimestre.setdefault((anio, trimestre), []).append(nombre)

claves_trimestres = sorted(grupos_por_trimestre.keys())
print(f"Trimestres a procesar: {len(claves_trimestres)}")
for (anio, trimestre) in claves_trimestres:
    print(f"  {anio}-T{trimestre}: {grupos_por_trimestre[(anio, trimestre)]}")

# ETAPA 3: Procesar cada trimestre -> resumen por estación y género
#          + conteo de estaciones activas por alcaldía
# -----------------------------------------------------------------------

trimestres_ya_procesados = cargar_estado(ESTADO_RESUMEN)
primera_escritura = not (os.path.exists(RUTA_SALIDA) and len(trimestres_ya_procesados) > 0)

archivos_con_columnas_faltantes = []

# Diagnóstico global: viajes que no se pudieron ubicar en ninguna alcaldía,
# y qué valores de estación aparecen más seguido entre esos no ubicados.
total_viajes_sin_catalogo = 0
total_viajes = 0
muestreo_estaciones_sin_catalogo = {}  # {valor_original_estacion: conteo}

for (anio, trimestre) in claves_trimestres:
    clave_trimestre = f"{anio}-T{trimestre}"

    # --- CHECKPOINT: saltar trimestres ya resumidos en una corrida anterior ---
    if clave_trimestre in trimestres_ya_procesados:
        print(f"{clave_trimestre}: ya procesado (checkpoint), se omite.")
        continue

    acumulado_viajes = None                     # Series (estacion, genero) -> n_viajes, se va sumando
    estaciones_activas_por_alcaldia = {}         # {alcaldia: set(clave_estacion)}

    for nombre in grupos_por_trimestre[(anio, trimestre)]:
        ruta = os.path.join(CARPETA_TEMP, nombre)

        mapa_cols, faltantes = resolver_columnas(ruta, COLUMNAS_NECESARIAS)
        if faltantes:
            columnas_reales = pq.ParquetFile(ruta).schema.names
            print(f"[AVISO] {nombre}: no se encontraron {faltantes}. "
                  f"Columnas reales: {columnas_reales}. Se omite este archivo (el trimestre "
                  f"se calcula con los meses restantes).")
            archivos_con_columnas_faltantes.append({
                "archivo": nombre, "faltantes": faltantes, "columnas_reales": columnas_reales,
            })
            continue

        df_mes = pd.read_parquet(ruta, columns=list(mapa_cols.values()))
        df_mes = df_mes.rename(columns={real: logico for logico, real in mapa_cols.items()})

        # --- Sumamos viajes de este mes al acumulado del trimestre ---
        # (se sigue agrupando por el valor ORIGINAL de la estación, para no
        # tocar lo que se expone después en el CSV de salida)
        parcial = df_mes.groupby([COL_ESTACION_ORIGEN, COL_GENERO]).size()
        acumulado_viajes = parcial if acumulado_viajes is None else acumulado_viajes.add(parcial, fill_value=0)

        # --- Registramos qué estaciones estuvieron activas este mes, por alcaldía ---
        # (aquí sí usamos la clave normalizada para encontrar la alcaldía correcta)
        estaciones_del_mes = df_mes[COL_ESTACION_ORIGEN].dropna().unique()
        for estacion in estaciones_del_mes:
            clave = normalizar_num_estacion(estacion)
            alcaldia = MAPA_ESTACION_ALCALDIA.get(clave, "SIN_CATALOGO")
            estaciones_activas_por_alcaldia.setdefault(alcaldia, set()).add(clave)

        del df_mes, parcial

    if acumulado_viajes is None:
        print(f"{clave_trimestre}: sin datos utilizables, se omite.")
        continue

    # Cada fila es (año, trimestre, estación, alcaldía, género) -- se
    # mantiene el detalle de ciclo_estacion_retiro (ya NO se colapsa a
    # nivel alcaldía), pero sí se le agrega la alcaldía correspondiente y
    # el conteo de estaciones activas de esa alcaldía en ese trimestre.
    resumen_trimestre = acumulado_viajes.reset_index(name="n_viajes")
    resumen_trimestre["n_viajes"] = resumen_trimestre["n_viajes"].astype(int)
    resumen_trimestre["anio"] = anio
    resumen_trimestre["trimestre"] = trimestre

    resumen_trimestre["alcaldia"] = resumen_trimestre[COL_ESTACION_ORIGEN].map(
        lambda e: MAPA_ESTACION_ALCALDIA.get(normalizar_num_estacion(e), "SIN_CATALOGO")
    )

    # Estaciones activas por alcaldía en este trimestre (un solo número por
    # alcaldía, repetido en todas las filas -- estación y género -- de esa
    # alcaldía, ya que es una propiedad de alcaldía+trimestre).
    resumen_trimestre["estaciones_activas_alcaldia"] = resumen_trimestre["alcaldia"].map(
        lambda a: len(estaciones_activas_por_alcaldia.get(a, set()))
    )

    columnas_orden = ["anio", "trimestre", COL_ESTACION_ORIGEN, "alcaldia",
                       COL_GENERO, "n_viajes", "estaciones_activas_alcaldia"]
    resumen_trimestre = resumen_trimestre[columnas_orden]

    resumen_trimestre.to_csv(
        RUTA_SALIDA,
        mode="a" if not primera_escritura else "w",
        header=primera_escritura,
        index=False,
    )
    primera_escritura = False

    fila_sin_catalogo = resumen_trimestre["alcaldia"] == "SIN_CATALOGO"
    n_sin_catalogo = int(resumen_trimestre.loc[fila_sin_catalogo, "n_viajes"].sum())
    n_total_trimestre = int(resumen_trimestre["n_viajes"].sum())

    print(f"{clave_trimestre}: {n_total_trimestre:,} viajes -> "
          f"{len(resumen_trimestre)} filas de resumen (estación x género) "
          f"({n_sin_catalogo:,} viajes de estaciones sin alcaldía conocida).")

    # Acumular diagnóstico global
    total_viajes += n_total_trimestre
    total_viajes_sin_catalogo += n_sin_catalogo
    for _, fila in resumen_trimestre.loc[fila_sin_catalogo].iterrows():
        valor = fila[COL_ESTACION_ORIGEN]
        muestreo_estaciones_sin_catalogo[valor] = muestreo_estaciones_sin_catalogo.get(valor, 0) + fila["n_viajes"]

    trimestres_ya_procesados.add(clave_trimestre)
    guardar_estado(ESTADO_RESUMEN, trimestres_ya_procesados)

    del acumulado_viajes, resumen_trimestre, estaciones_activas_por_alcaldia

print(f"\nProceso terminado. Resumen por trimestre guardado en: {RUTA_SALIDA}")

if archivos_con_columnas_faltantes:
    print(f"\n[AVISO] {len(archivos_con_columnas_faltantes)} archivo(s) individuales se omitieron "
          f"por columnas no encontradas (el trimestre correspondiente igual se calculó con los "
          f"meses restantes disponibles):")
    for a in archivos_con_columnas_faltantes:
        print(f"  - {a['archivo']}: faltan {a['faltantes']} | columnas reales: {a['columnas_reales']}")

# --- Diagnóstico final del cruce estación -> alcaldía ---
if total_viajes:
    pct_sin_catalogo = 100 * total_viajes_sin_catalogo / total_viajes
    print(f"\n[DIAGNÓSTICO DEL CRUCE] {total_viajes_sin_catalogo:,} de {total_viajes:,} viajes "
          f"({pct_sin_catalogo:.1f}%) quedaron como SIN_CATALOGO en esta corrida.")
    if muestreo_estaciones_sin_catalogo:
        top_sin_catalogo = sorted(muestreo_estaciones_sin_catalogo.items(), key=lambda x: -x[1])[:15]
        print("Valores de estación más frecuentes SIN alcaldía conocida (revisa si existen "
              "en tu catálogo con otro formato, o si son estaciones que ya no operan):")
        for valor, n in top_sin_catalogo:
            print(f"   {valor!r}: {n:,} viajes")

Catálogo cargado: 677 estaciones con alcaldía conocida (de 677 filas originales).
Trimestres a procesar: 27
  2020-T1: ['2020-01.parquet', '2020-02.parquet', '2020-03.parquet']
  2020-T2: ['2020-04.parquet', '2020-05.parquet', '2020-06.parquet']
  2020-T3: ['2020-07.parquet', '2020-08.parquet', '2020-09.parquet']
  2020-T4: ['2020-10.parquet', '2020-11.parquet', '2020-12.parquet']
  2021-T1: ['2021-01.parquet', '2021-02.parquet', '2021-03.parquet']
  2021-T2: ['2021-04.parquet', '2021-05.parquet', '2021-06.parquet']
  2021-T3: ['2021-07.parquet', '2021-08.parquet', '2021-09.parquet']
  2021-T4: ['2021-10.parquet', '2021-11.parquet', '2021-12.parquet']
  2022-T1: ['2022-01.parquet', '2022-02.parquet', '2022-03.parquet']
  2022-T2: ['2022-04.parquet', '2022-05.parquet', '2022-06.parquet']
  2022-T3: ['2022-07.parquet', '2022-08.parquet', '2022-09.parquet']
  2022-T4: ['2022-10.parquet', '2022-11.parquet', '2022-12.parquet']
  2023-T1: ['2023-01.parquet', '2023-02.parquet', '2023-03.parqu

/tmp/ipykernel_748/3798636941.py:253: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  acumulado_viajes = parcial if acumulado_viajes is None else acumulado_viajes.add(parcial, fill_value=0)
/tmp/ipykernel_748/3798636941.py:253: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  acumulado_viajes = parcial if acumulado_viajes is None else acumulado_viajes.add(parcial, fill_value=0)


2022-T3: 1,089,569 viajes -> 1837 filas de resumen (estación x género) (69,419 viajes de estaciones sin alcaldía conocida).
2022-T4: 1,085,968 viajes -> 744 filas de resumen (estación x género) (0 viajes de estaciones sin alcaldía conocida).
2023-T1: 2,069,329 viajes -> 1223 filas de resumen (estación x género) (18 viajes de estaciones sin alcaldía conocida).
2023-T2: 2,768,768 viajes -> 1448 filas de resumen (estación x género) (31 viajes de estaciones sin alcaldía conocida).
2023-T3: 3,298,540 viajes -> 1625 filas de resumen (estación x género) (12 viajes de estaciones sin alcaldía conocida).
2023-T4: 4,040,761 viajes -> 1897 filas de resumen (estación x género) (415 viajes de estaciones sin alcaldía conocida).
2024-T1: 5,087,255 viajes -> 2014 filas de resumen (estación x género) (0 viajes de estaciones sin alcaldía conocida).
2024-T2: 5,908,705 viajes -> 2034 filas de resumen (estación x género) (4 viajes de estaciones sin alcaldía conocida).
2024-T3: 5,569,107 viajes -> 2055 filas

In [34]:
valor_crudo = df_catalogo.loc[df_catalogo["alcaldia"].notna(), "short_name"]
print(valor_crudo[valor_crudo.astype(str).str.contains("99")].head(10))
print(df_catalogo["short_name"].dtype)

47     299
136    399
184    599
267    699
376    099
545    199
649    499
Name: short_name, dtype: object
object
